In [1]:
import pandas as pd, numpy as np, os
print(os.getcwd())      # should end in ...\nifty50-direction-forecasting
d = pd.read_csv("nifty50_features.csv", parse_dates=["Date"])
print(d.shape, d["Date"].min().date(), d["Date"].max().date())
print(d["crude_chg"].min(), d["crude_chg"].max())

C:\Users\Kaif\projects\nifty50-direction-forecasting
(4071, 26) 2010-03-17 2026-10-12
-305.9660652724544 39.53717822647376


In [3]:
bad = (d["crude"] <= 0) | (d["crude"].shift() <= 0)   
d.loc[bad, "crude_chg"] = 0
d["crude_chg"] = d["crude_chg"].clip(-20, 20)

print(d["crude_chg"].min(), d["crude_chg"].max())   
d.to_csv("nifty50_features.csv", index=False)

-20.0 20.0


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

features = ["ret", "ret_lag1", "ret_lag2", "ret_lag3", "ret_lag5",
            "ma10_ratio", "ma50_ratio", "vol_21d", "rsi14",
            "vix", "vix_chg", "usdinr_chg", "sp500_chg", "crude_chg", "dow"]

train = d[d["Date"] < "2020-01-01"]
val   = d[(d["Date"] >= "2020-01-01") & (d["Date"] < "2023-01-01")]
test  = d[d["Date"] >= "2023-01-01"]      # leave untouched until the very end

print(len(train), len(val), len(test))
y_val = val["target_up"]
print("Always up:     ", round(y_val.mean(), 3))
print("Same as today: ", round(accuracy_score(y_val, (val["ret"] > 0).astype(int)), 3))

2394 746 931
Always up:      0.548
Same as today:  0.568


In [7]:
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=50, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=150, max_depth=2, learning_rate=0.05, random_state=42),
}

rows = []
for name, m in models.items():
    m.fit(train[features], train["target_up"])
    pred = m.predict(val[features])
    proba = m.predict_proba(val[features])[:, 1]
    rows.append({"model": name,
                 "accuracy": accuracy_score(y_val, pred),
                 "precision": precision_score(y_val, pred, zero_division=0),
                 "recall": recall_score(y_val, pred),
                 "roc_auc": roc_auc_score(y_val, proba)})
print(pd.DataFrame(rows).round(3))

                 model  accuracy  precision  recall  roc_auc
0  Logistic Regression     0.635      0.667   0.667    0.674
1        Random Forest     0.638      0.662   0.694    0.662
2    Gradient Boosting     0.635      0.667   0.670    0.670


In [9]:
overnight = ["sp500_chg", "crude_chg", "usdinr_chg"]
f_clean = [f for f in features if f not in overnight]

for name, cols in [("with overnight features", features), ("without overnight features", f_clean)]:
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    m.fit(train[cols], train["target_up"])
    p = m.predict_proba(val[cols])[:, 1]
    print(name, "| accuracy:", round(accuracy_score(y_val, (p > 0.5).astype(int)), 3),
          "| AUC:", round(roc_auc_score(y_val, p), 3))

with overnight features | accuracy: 0.635 | AUC: 0.674
without overnight features | accuracy: 0.548 | AUC: 0.529


In [11]:
d["gap_up"]      = (d["Open"].shift(-1) > d["Close"]).astype(int)            # next open vs today's close
d["intraday_up"] = (d["Close"].shift(-1) > d["Open"].shift(-1)).astype(int)  # next close vs next open
d2 = d.iloc[:-1].copy()                                                       # last row has no tomorrow

train2 = d2[d2["Date"] < "2020-01-01"]
val2   = d2[(d2["Date"] >= "2020-01-01") & (d2["Date"] < "2023-01-01")]

for target in ["gap_up", "intraday_up"]:
    print(f"\n{target}: always-up baseline = {val2[target].mean():.3f}")
    for name, cols in [("all features", features), ("no overnight features", f_clean)]:
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
        m.fit(train2[cols], train2[target])
        p = m.predict_proba(val2[cols])[:, 1]
        print(f"  {name:22s} accuracy {accuracy_score(val2[target], (p > 0.5).astype(int)):.3f} | AUC {roc_auc_score(val2[target], p):.3f}")


gap_up: always-up baseline = 0.680
  all features           accuracy 0.771 | AUC 0.790
  no overnight features  accuracy 0.651 | AUC 0.529

intraday_up: always-up baseline = 0.504
  all features           accuracy 0.507 | AUC 0.508
  no overnight features  accuracy 0.493 | AUC 0.504


In [13]:
trainval = d2[d2["Date"] < "2023-01-01"]
test2    = d2[d2["Date"] >= "2023-01-01"]

for target in ["gap_up", "intraday_up"]:
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    m.fit(trainval[features], trainval[target])
    p = m.predict_proba(test2[features])[:, 1]
    print(f"{target}: baseline {test2[target].mean():.3f} | "
          f"accuracy {accuracy_score(test2[target], (p > 0.5).astype(int)):.3f} | "
          f"AUC {roc_auc_score(test2[target], p):.3f}")

gap_up: baseline 0.611 | accuracy 0.671 | AUC 0.732
intraday_up: baseline 0.470 | accuracy 0.508 | AUC 0.472
